In [ ]:
!pip install pandas rapidfuzz faker streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 55.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import random
from faker import Faker
from datetime import timedelta

fake = Faker()

# Make results reproducible
random.seed(42)
Faker.seed(42)

# Number of transactions
n = 60

records = []

for i in range(n):
    txn_id = f"TXN{1000+i}"
    amount = round(random.uniform(500, 20000), 2)
    date = fake.date_between(start_date="-30d", end_date="today")

    records.append({
        "txn_id": txn_id,
        "amount": amount,
        "date": date
    })

ledger = pd.DataFrame(records)

# Save ground truth
ledger.to_csv("ledger.csv", index=False)

print("Ground truth transactions created:", len(ledger))

ledger.head(10)

Ground truth transactions created: 60


,txn_id,amount,date
0,TXN1000,12968.82,2026-08-23
1,TXN1001,987.71,2026-08-04
2,TXN1002,5863.07,2026-08-12
3,TXN1003,4852.61,2026-08-10
4,TXN1004,14861.19,2026-08-26
5,TXN1005,13695.64,2026-08-24
6,TXN1006,17897.50,2026-08-30
7,TXN1007,2195.31,2026-08-06
8,TXN1008,8727.48,2026-08-16
9,TXN1009,1081.05,2026-08-04


In [ ]:
import copy

bank_records = []

# Keep track of what happened to each transaction
ground_truth_mapping = {}

for r in records:

    original_txn_id = r["txn_id"]

    r = copy.deepcopy(r)

    roll = random.random()

    # 10% disappear from the bank statement
    if roll < 0.10:
        ground_truth_mapping[original_txn_id] = {
            "status": "missing_from_bank"
        }
        continue

    # 10% have a fee deduction
    elif roll < 0.20:
        original_amount = r["amount"]
        r["amount"] = round(r["amount"] * 0.98, 2)

        ground_truth_mapping[original_txn_id] = {
            "status": "fee_deduction",
            "bank_txn_id": r["txn_id"]
        }

    # 10% have settlement delay
    elif roll < 0.30:
        r["date"] = r["date"] + timedelta(
            days=random.choice([1, 2])
        )

        ground_truth_mapping[original_txn_id] = {
            "status": "settlement_delay",
            "bank_txn_id": r["txn_id"]
        }

    # 5% have corrupted transaction ID
    elif roll < 0.35:
        r["txn_id"] = r["txn_id"][:-1] + "X"

        ground_truth_mapping[original_txn_id] = {
            "status": "corrupted_reference",
            "bank_txn_id": r["txn_id"]
        }

    # Remaining transactions remain unchanged
    else:
        ground_truth_mapping[original_txn_id] = {
            "status": "exact",
            "bank_txn_id": r["txn_id"]
        }

    bank_records.append(r)


# Add duplicate bank transactions
for _ in range(3):
    bank_records.append(
        copy.deepcopy(random.choice(bank_records))
    )


# Add bank-only transactions
for i in range(3):

    bank_only_id = f"BANKONLY{i}"

    bank_records.append({
        "txn_id": bank_only_id,
        "amount": round(random.uniform(500, 5000), 2),
        "date": fake.date_between(
            start_date="-30d",
            end_date="today"
        )
    })


bank = pd.DataFrame(bank_records)

bank.to_csv("bank_statement.csv", index=False)

print("Ledger records:", len(ledger))
print("Bank records:", len(bank))

bank.head(10)

Ledger records: 60
Bank records: 65


,txn_id,amount,date
0,TXN1000,12968.82,2026-08-23
1,TXN1001,987.71,2026-08-04
2,TXN1002,5863.07,2026-08-12
3,TXN1003,4852.61,2026-08-10
4,TXN1004,14861.19,2026-08-26
5,TXN1005,13695.64,2026-08-24
6,TXN1006,17897.50,2026-08-31
7,TXN1007,2195.31,2026-08-06
8,TXN1008,8727.48,2026-08-16
9,TXN1010,4763.44,2026-08-10


## Comparing the Ledger and Bank Statement

The ledger represents the merchant's internal records.

The bank statement represents the transactions actually appearing in the bank.

The bank data has intentionally introduced realistic discrepancies such as:
- Missing transactions
- Fee deductions
- Settlement delays
- Corrupted transaction references
- Duplicate transactions
- Bank-only transactions


In [ ]:
print("LEDGER")
display(ledger.head(10))

print("\nBANK STATEMENT")
display(bank.head(10))


LEDGER


,txn_id,amount,date
0,TXN1000,12968.82,2026-08-23
1,TXN1001,987.71,2026-08-04
2,TXN1002,5863.07,2026-08-12
3,TXN1003,4852.61,2026-08-10
4,TXN1004,14861.19,2026-08-26
5,TXN1005,13695.64,2026-08-24
6,TXN1006,17897.50,2026-08-30
7,TXN1007,2195.31,2026-08-06
8,TXN1008,8727.48,2026-08-16
9,TXN1009,1081.05,2026-08-04



BANK STATEMENT


,txn_id,amount,date
0,TXN1000,12968.82,2026-08-23
1,TXN1001,987.71,2026-08-04
2,TXN1002,5863.07,2026-08-12
3,TXN1003,4852.61,2026-08-10
4,TXN1004,14861.19,2026-08-26
5,TXN1005,13695.64,2026-08-24
6,TXN1006,17897.50,2026-08-31
7,TXN1007,2195.31,2026-08-06
8,TXN1008,8727.48,2026-08-16
9,TXN1010,4763.44,2026-08-10


In [ ]:
from rapidfuzz import fuzz


def match_records(
    ledger,
    bank,
    amount_tol=0.05,
    date_tol_days=2,
    fuzzy_thresh=85
):

    ledger = ledger.copy()
    bank = bank.copy()

    ledger["date"] = pd.to_datetime(ledger["date"])
    bank["date"] = pd.to_datetime(bank["date"])

    matches = []

    used_bank_idx = set()

    # --------------------------------------------------
    # TIER 1: EXACT MATCH
    # --------------------------------------------------

    for i, l in ledger.iterrows():

        for j, b in bank.iterrows():

            if j in used_bank_idx:
                continue

            if (
                l["txn_id"] == b["txn_id"]
                and l["amount"] == b["amount"]
                and l["date"] == b["date"]
            ):

                matches.append({
                    "ledger_txn_id": l["txn_id"],
                    "bank_txn_id": b["txn_id"],
                    "ledger_amount": l["amount"],
                    "bank_amount": b["amount"],
                    "ledger_date": l["date"],
                    "bank_date": b["date"],
                    "amount_difference": 0,
                    "date_difference_days": 0,
                    "tier": "exact",
                    "confidence": 1.00,
                    "reason": "Transaction ID, amount and date all match"
                })

                used_bank_idx.add(j)

                break


    matched_ledger_ids = {
        m["ledger_txn_id"] for m in matches
    }


    # --------------------------------------------------
    # TIER 2: SAME ID + SMALL DIFFERENCES
    # --------------------------------------------------

    for i, l in ledger.iterrows():

        if l["txn_id"] in matched_ledger_ids:
            continue

        for j, b in bank.iterrows():

            if j in used_bank_idx:
                continue

            if l["txn_id"] == b["txn_id"]:

                amount_difference = abs(
                    l["amount"] - b["amount"]
                ) / l["amount"]

                date_difference = abs(
                    (l["date"] - b["date"]).days
                )

                if (
                    amount_difference <= amount_tol
                    and date_difference <= date_tol_days
                ):

                    matches.append({
                        "ledger_txn_id": l["txn_id"],
                        "bank_txn_id": b["txn_id"],
                        "ledger_amount": l["amount"],
                        "bank_amount": b["amount"],
                        "ledger_date": l["date"],
                        "bank_date": b["date"],
                        "amount_difference": round(
                            amount_difference * 100, 2
                        ),
                        "date_difference_days": date_difference,
                        "tier": "tolerance",
                        "confidence": 0.90,
                        "reason": "Same transaction ID with small amount/date difference"
                    })

                    used_bank_idx.add(j)
                    matched_ledger_ids.add(l["txn_id"])

                    break


    # --------------------------------------------------
    # TIER 3: FUZZY MATCH
    # --------------------------------------------------

    for i, l in ledger.iterrows():

        if l["txn_id"] in matched_ledger_ids:
            continue

        best_score = 0
        best_j = None
        best_amount_difference = None
        best_date_difference = None

        for j, b in bank.iterrows():

            if j in used_bank_idx:
                continue

            similarity = fuzz.ratio(
                l["txn_id"],
                b["txn_id"]
            )

            amount_difference = abs(
                l["amount"] - b["amount"]
            ) / l["amount"]

            date_difference = abs(
                (l["date"] - b["date"]).days
            )

            if (
                similarity >= fuzzy_thresh
                and amount_difference <= amount_tol
                and date_difference <= date_tol_days
            ):

                if similarity > best_score:

                    best_score = similarity
                    best_j = j
                    best_amount_difference = amount_difference
                    best_date_difference = date_difference


        if best_j is not None:

            b = bank.loc[best_j]

            matches.append({
                "ledger_txn_id": l["txn_id"],
                "bank_txn_id": b["txn_id"],
                "ledger_amount": l["amount"],
                "bank_amount": b["amount"],
                "ledger_date": l["date"],
                "bank_date": b["date"],
                "amount_difference": round(
                    best_amount_difference * 100, 2
                ),
                "date_difference_days": best_date_difference,
                "tier": "fuzzy",
                "confidence": round(best_score / 100, 2),
                "reason": f"Reference similarity = {best_score:.1f}%"
            })

            used_bank_idx.add(best_j)
            matched_ledger_ids.add(l["txn_id"])


    # --------------------------------------------------
    # CREATE OUTPUT TABLES
    # --------------------------------------------------

    matched_df = pd.DataFrame(matches)

    unmatched_ledger = ledger[
        ~ledger["txn_id"].isin(matched_ledger_ids)
    ].copy()

    unmatched_bank = bank[
        ~bank.index.isin(used_bank_idx)
    ].copy()


    return (
        matched_df,
        unmatched_ledger,
        unmatched_bank
    )

In [ ]:
matched, unmatched_ledger, unmatched_bank = match_records(
    ledger,
    bank
)

print("Reconciliation complete!")

print("\nMatched:", len(matched))
print("Unmatched ledger:", len(unmatched_ledger))
print("Unmatched bank:", len(unmatched_bank))

Reconciliation complete!

Matched: 59
Unmatched ledger: 1
Unmatched bank: 6


In [ ]:
matched.head(10)

,ledger_txn_id,bank_txn_id,ledger_amount,bank_amount,ledger_date,bank_date,amount_difference,date_difference_days,tier,confidence,reason
0,TXN1000,TXN1000,12968.82,12968.82,2026-08-23,2026-08-23,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
1,TXN1001,TXN1001,987.71,987.71,2026-08-04,2026-08-04,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
2,TXN1002,TXN1002,5863.07,5863.07,2026-08-12,2026-08-12,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
3,TXN1003,TXN1003,4852.61,4852.61,2026-08-10,2026-08-10,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
4,TXN1004,TXN1004,14861.19,14861.19,2026-08-26,2026-08-26,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
5,TXN1005,TXN1005,13695.64,13695.64,2026-08-24,2026-08-24,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
6,TXN1007,TXN1007,2195.31,2195.31,2026-08-06,2026-08-06,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
7,TXN1008,TXN1008,8727.48,8727.48,2026-08-16,2026-08-16,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
8,TXN1010,TXN1010,4763.44,4763.44,2026-08-10,2026-08-10,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
9,TXN1011,TXN1011,10354.43,10354.43,2026-08-19,2026-08-19,0.0,0,exact,1.0,"Transaction ID, amount and date all match"


In [ ]:
print("MATCH TYPE BREAKDOWN")
print(matched["tier"].value_counts())


MATCH TYPE BREAKDOWN
tier
exact        45
tolerance    13
fuzzy         1
Name: count, dtype: int64


In [ ]:
print("UNMATCHED LEDGER TRANSACTIONS")
display(unmatched_ledger)

UNMATCHED LEDGER TRANSACTIONS


,txn_id,amount,date
9,TXN1009,1081.05,2026-08-04


In [ ]:
print("UNMATCHED BANK TRANSACTIONS")
display(unmatched_bank)

UNMATCHED BANK TRANSACTIONS


,txn_id,amount,date
59,TXN1032,10956.45,2026-08-20
60,TXN1001,987.71,2026-08-04
61,TXN1007,2195.31,2026-08-06
62,BANKONLY0,4680.94,2026-09-02
63,BANKONLY1,4454.25,2026-08-23
64,BANKONLY2,4242.49,2026-08-20


In [ ]:
matched[matched["tier"] != "exact"]

,ledger_txn_id,bank_txn_id,ledger_amount,bank_amount,ledger_date,bank_date,amount_difference,date_difference_days,tier,confidence,reason
45,TXN1006,TXN1006,17897.50,17897.50,2026-08-30,2026-08-31,0.0,1,tolerance,0.90,Same transaction ID with small amount/date dif...
46,TXN1013,TXN1013,4377.33,4377.33,2026-08-09,2026-08-11,0.0,2,tolerance,0.90,Same transaction ID with small amount/date dif...
47,TXN1017,TXN1017,11990.68,11990.68,2026-08-21,2026-08-22,0.0,1,tolerance,0.90,Same transaction ID with small amount/date dif...
48,TXN1025,TXN1025,7063.59,6922.32,2026-08-14,2026-08-14,2.0,0,tolerance,0.90,Same transaction ID with small amount/date dif...
49,TXN1029,TXN1029,12272.66,12027.21,2026-08-22,2026-08-22,2.0,0,tolerance,0.90,Same transaction ID with small amount/date dif...
50,TXN1030,TXN1030,16239.00,15914.22,2026-08-28,2026-08-28,2.0,0,tolerance,0.90,Same transaction ID with small amount/date dif...
51,TXN1036,TXN1036,16673.39,16673.39,2026-08-28,2026-08-29,0.0,1,tolerance,0.90,Same transaction ID with small amount/date dif...
52,TXN1038,TXN1038,17303.28,16957.21,2026-08-29,2026-08-29,2.0,0,tolerance,0.90,Same transaction ID with small amount/date dif...
53,TXN1043,TXN1043,6143.07,6143.07,2026-08-12,2026-08-13,0.0,1,tolerance,0.90,Same transaction ID with small amount/date dif...
54,TXN1047,TXN1047,5920.49,5920.49,2026-08-12,2026-08-13,0.0,1,tolerance,0.90,Same transaction ID with small amount/date dif...


In [ ]:
matched, unmatched_ledger, unmatched_bank = match_records(
    ledger,
    bank
)

print("Reconciliation complete!")

print("\nMatched:", len(matched))
print("Unmatched ledger:", len(unmatched_ledger))
print("Unmatched bank:", len(unmatched_bank))

Reconciliation complete!

Matched: 59
Unmatched ledger: 1
Unmatched bank: 6


In [ ]:
matched.head(10)

,ledger_txn_id,bank_txn_id,ledger_amount,bank_amount,ledger_date,bank_date,amount_difference,date_difference_days,tier,confidence,reason
0,TXN1000,TXN1000,12968.82,12968.82,2026-08-23,2026-08-23,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
1,TXN1001,TXN1001,987.71,987.71,2026-08-04,2026-08-04,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
2,TXN1002,TXN1002,5863.07,5863.07,2026-08-12,2026-08-12,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
3,TXN1003,TXN1003,4852.61,4852.61,2026-08-10,2026-08-10,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
4,TXN1004,TXN1004,14861.19,14861.19,2026-08-26,2026-08-26,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
5,TXN1005,TXN1005,13695.64,13695.64,2026-08-24,2026-08-24,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
6,TXN1007,TXN1007,2195.31,2195.31,2026-08-06,2026-08-06,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
7,TXN1008,TXN1008,8727.48,8727.48,2026-08-16,2026-08-16,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
8,TXN1010,TXN1010,4763.44,4763.44,2026-08-10,2026-08-10,0.0,0,exact,1.0,"Transaction ID, amount and date all match"
9,TXN1011,TXN1011,10354.43,10354.43,2026-08-19,2026-08-19,0.0,0,exact,1.0,"Transaction ID, amount and date all match"


In [ ]:
total_ledger = len(ledger)
total_matched = len(matched)

match_rate = total_matched / total_ledger

print(f"Total ledger transactions: {total_ledger}")
print(f"Automatically matched: {total_matched}")
print(f"Match rate: {match_rate:.2%}")

Total ledger transactions: 60
Automatically matched: 59
Match rate: 98.33%


In [ ]:
if len(matched) > 0:

    print("Match breakdown:")

    print(
        matched["tier"].value_counts()
    )

Match breakdown:
tier
exact        45
tolerance    13
fuzzy         1
Name: count, dtype: int64


In [ ]:
exceptions = []

# Ledger transactions with no bank match
for _, row in unmatched_ledger.iterrows():

    exceptions.append({
        "txn_id": row["txn_id"],
        "amount": row["amount"],
        "date": row["date"],
        "side": "ledger",
        "reason": "No corresponding bank transaction found"
    })


# Bank transactions with no ledger match
for _, row in unmatched_bank.iterrows():

    exceptions.append({
        "txn_id": row["txn_id"],
        "amount": row["amount"],
        "date": row["date"],
        "side": "bank",
        "reason": "No corresponding ledger transaction found"
    })


exceptions_df = pd.DataFrame(exceptions)

exceptions_df

,txn_id,amount,date,side,reason
0,TXN1009,1081.05,2026-08-04,ledger,No corresponding bank transaction found
1,TXN1032,10956.45,2026-08-20,bank,No corresponding ledger transaction found
2,TXN1001,987.71,2026-08-04,bank,No corresponding ledger transaction found
3,TXN1007,2195.31,2026-08-06,bank,No corresponding ledger transaction found
4,BANKONLY0,4680.94,2026-09-02,bank,No corresponding ledger transaction found
5,BANKONLY1,4454.25,2026-08-23,bank,No corresponding ledger transaction found
6,BANKONLY2,4242.49,2026-08-20,bank,No corresponding ledger transaction found


In [ ]:
if len(exceptions_df) > 0:

    exceptions_df["review_required"] = True

exceptions_df

,txn_id,amount,date,side,reason,review_required
0,TXN1009,1081.05,2026-08-04,ledger,No corresponding bank transaction found,True
1,TXN1032,10956.45,2026-08-20,bank,No corresponding ledger transaction found,True
2,TXN1001,987.71,2026-08-04,bank,No corresponding ledger transaction found,True
3,TXN1007,2195.31,2026-08-06,bank,No corresponding ledger transaction found,True
4,BANKONLY0,4680.94,2026-09-02,bank,No corresponding ledger transaction found,True
5,BANKONLY1,4454.25,2026-08-23,bank,No corresponding ledger transaction found,True
6,BANKONLY2,4242.49,2026-08-20,bank,No corresponding ledger transaction found,True


In [ ]:
matched.to_csv("matched.csv", index=False)

unmatched_ledger.to_csv(
    "unmatched_ledger.csv",
    index=False
)

unmatched_bank.to_csv(
    "unmatched_bank.csv",
    index=False
)

exceptions_df.to_csv(
    "exceptions.csv",
    index=False
)

print("All result files saved!")

All result files saved!


In [ ]:
audit_columns = [
    "ledger_txn_id",
    "bank_txn_id",
    "tier",
    "confidence",
    "amount_difference",
    "date_difference_days",
    "reason"
]

matched[audit_columns].head(20)

,ledger_txn_id,bank_txn_id,tier,confidence,amount_difference,date_difference_days,reason
0,TXN1000,TXN1000,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
1,TXN1001,TXN1001,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
2,TXN1002,TXN1002,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
3,TXN1003,TXN1003,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
4,TXN1004,TXN1004,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
5,TXN1005,TXN1005,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
6,TXN1007,TXN1007,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
7,TXN1008,TXN1008,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
8,TXN1010,TXN1010,exact,1.0,0.0,0,"Transaction ID, amount and date all match"
9,TXN1011,TXN1011,exact,1.0,0.0,0,"Transaction ID, amount and date all match"


In [ ]:
if len(matched) > 0:

    print("Match breakdown:")

    print(
        matched["tier"].value_counts()
    )

Match breakdown:
tier
exact        45
tolerance    13
fuzzy         1
Name: count, dtype: int64


In [ ]:
print(f"Match rate: {len(matched)}/{len(ledger)} = {len(matched)/len(ledger):.2%}")

Match rate: 59/60 = 98.33%


In [ ]:
matched[matched["tier"] == "fuzzy"][audit_columns]

,ledger_txn_id,bank_txn_id,tier,confidence,amount_difference,date_difference_days,reason
58,TXN1042,TXN104X,fuzzy,0.86,0.0,0,Reference similarity = 85.7%


In [ ]:
exceptions = []

for _, row in unmatched_ledger.iterrows():
    exceptions.append({
        "txn_id": row["txn_id"],
        "amount": row["amount"],
        "date": row["date"],
        "side": "ledger",
        "reason": "No corresponding bank transaction found"
    })

for _, row in unmatched_bank.iterrows():
    exceptions.append({
        "txn_id": row["txn_id"],
        "amount": row["amount"],
        "date": row["date"],
        "side": "bank",
        "reason": "No corresponding ledger transaction found"
    })

exceptions_df = pd.DataFrame(exceptions)
exceptions_df

,txn_id,amount,date,side,reason
0,TXN1009,1081.05,2026-08-04,ledger,No corresponding bank transaction found
1,TXN1032,10956.45,2026-08-20,bank,No corresponding ledger transaction found
2,TXN1001,987.71,2026-08-04,bank,No corresponding ledger transaction found
3,TXN1007,2195.31,2026-08-06,bank,No corresponding ledger transaction found
4,BANKONLY0,4680.94,2026-09-02,bank,No corresponding ledger transaction found
5,BANKONLY1,4454.25,2026-08-23,bank,No corresponding ledger transaction found
6,BANKONLY2,4242.49,2026-08-20,bank,No corresponding ledger transaction found


In [ ]:
# Tag likely duplicates specifically
matched_bank_ids = set(matched["bank_txn_id"])

for exc in exceptions:
    if exc["side"] == "bank" and exc["txn_id"] in matched_bank_ids:
        exc["reason"] = "Possible duplicate bank entry (transaction ID already matched elsewhere)"

exceptions_df = pd.DataFrame(exceptions)
exceptions_df

,txn_id,amount,date,side,reason
0,TXN1009,1081.05,2026-08-04,ledger,No corresponding bank transaction found
1,TXN1032,10956.45,2026-08-20,bank,Possible duplicate bank entry (transaction ID ...
2,TXN1001,987.71,2026-08-04,bank,Possible duplicate bank entry (transaction ID ...
3,TXN1007,2195.31,2026-08-06,bank,Possible duplicate bank entry (transaction ID ...
4,BANKONLY0,4680.94,2026-09-02,bank,No corresponding ledger transaction found
5,BANKONLY1,4454.25,2026-08-23,bank,No corresponding ledger transaction found
6,BANKONLY2,4242.49,2026-08-20,bank,No corresponding ledger transaction found


In [ ]:
matched.to_csv("matched.csv", index=False)
unmatched_ledger.to_csv("unmatched_ledger.csv", index=False)
unmatched_bank.to_csv("unmatched_bank.csv", index=False)
exceptions_df.to_csv("exceptions.csv", index=False)

print("All result files saved!")

All result files saved!


In [ ]:
print("=" * 50)
print("RECONCILIATION AUDIT SUMMARY")
print("=" * 50)
print(f"Total ledger transactions: {len(ledger)}")
print(f"Total bank transactions: {len(bank)}")
print(f"Matched: {len(matched)} ({len(matched)/len(ledger):.2%})")
print()
print("Match tier breakdown:")
print(matched["tier"].value_counts().to_string())
print()
print(f"Exceptions requiring human review: {len(exceptions_df)}")
print(exceptions_df["reason"].value_counts().to_string())
print("=" * 50)

RECONCILIATION AUDIT SUMMARY
Total ledger transactions: 60
Total bank transactions: 65
Matched: 59 (98.33%)

Match tier breakdown:
tier
exact        45
tolerance    13
fuzzy         1

Exceptions requiring human review: 7
reason
Possible duplicate bank entry (transaction ID already matched elsewhere)    3
No corresponding ledger transaction found                                   3
No corresponding bank transaction found                                     1
